#### Strategic Parameters

In [1]:
import pandas as pd
import numpy as np

df = pd.read_csv('../data/processed/romina_market_linked_v2.csv')

EUDR_PREMIUM_UPLIFT = 1.22  # 22% extra for verified specialty
QUALITY_PREMIUM_KG = 0.50   # Extra $/kg for Grade 1 Sidama/Yirgacheffe

print("EUDR Strategy Engine Online.")

EUDR Strategy Engine Online.


#### The "Revenue Maximizer" Logic

In [2]:
def calculate_optimized_revenue(row):
    base_net = row['Net_Revenue_USD']
    
    specialty_stations = ["Astatke Menafesha", "Belete Tanga", "Zenebe Dikicha", "Dogada Yemasira", "Michael Girma"]
    
    if row['Status'] == 'Passed' and row['EUDR_Verified'] == True:
        if row['Station'] in specialty_stations:
            return (base_net * EUDR_PREMIUM_UPLIFT) + (row['Exportable_kg'] * QUALITY_PREMIUM_KG)
        else:
            return base_net * 1.07
            
    return base_net

# Apply the optimization
df['Optimized_Net_USD'] = df.apply(calculate_optimized_revenue, axis=1)

# Calculate the Gain
df['Premium_Gain_USD'] = df['Optimized_Net_USD'] - df['Net_Revenue_USD']

print(f"Total Potential Value Added by Strategy: ${df['Premium_Gain_USD'].sum():,.2f}")

Total Potential Value Added by Strategy: $7,749,180.96


#### Station-Level Performance

In [4]:
station_performance = df.groupby('Station').agg({
    'Lot_ID': 'count',
    'EUDR_Verified': lambda x: (x == True).mean() * 100,
    'Premium_Gain_USD': 'sum',
    'Status': lambda x: (x == 'Rejected').sum()
}).rename(columns={
    'Lot_ID': 'Total_Lots',
    'EUDR_Verified': 'Compliance_Score',
    'Status': 'Rejected_Count'
})

station_performance = station_performance.sort_values('Premium_Gain_USD', ascending=False)
print("STATION REVENUE LEADERBOARD")
station_performance.head(10)

STATION REVENUE LEADERBOARD


,Total_Lots,Compliance_Score,Premium_Gain_USD,Rejected_Count
Station,,,,
Belete Tanga,54,77.777778,1.466167e+06,14
Michael Girma,54,72.222222,1.418105e+06,10
Dogada Yemasira,49,81.632653,1.329727e+06,12
Zenebe Dikicha,49,79.591837,1.308130e+06,11
Astatke Menafesha,50,76.000000,1.293089e+06,11
Hagire Buticha,53,77.358491,3.464803e+05,16
Abebe Jengelo,49,75.510204,3.161016e+05,12
Oddi Boku,42,71.428571,2.713801e+05,7


In [5]:
df['Optimized_Retained_Forex'] = df['Optimized_Net_USD'] * 0.50

df.to_csv('../data/processed/romina_final_audit_v2.csv', index=False)
print("Master Data ready")

Master Data ready
